# SpaceX Falcon 9 - Data Collection (Web Scraping)

Scrape the Wikipedia "List of Falcon 9 and Falcon Heavy launches" page with BeautifulSoup to cross-check and enrich the API-collected records (exact dates, outcomes, booster versions, launch site names).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

## Locate the launch-records table and extract column headers

In [ ]:
html_tables = soup.find_all('table', {'class': 'wikitable'})
first_launch_table = html_tables[2]

column_names = []
def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colname = ' '.join(row.contents)
    if not colname.strip().isdigit():
        return colname.strip()

for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

## Parse each table row into a launch record

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass = mass[0:mass.find("kg") + 2]
    else:
        mass = 0
    return mass

launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [ ]:
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])
            launch_dict['Version Booster'].append(booster_version(row[1]))
            launch_dict['Launch site'].append(row[2].a.string if row[2].a else row[2].get_text(strip=True))
            launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].get_text(strip=True))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].get_text(strip=True))
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].get_text(strip=True))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))

print(f"Parsed {extracted_row} launch rows")

In [ ]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.to_csv('spacex_web_scraped.csv', index=False)
df.head()